# Experiment 2 — Baseline Validity

**PDF reference: Phase 2, Experiment 2**

Validates the spline baseline model fit on the reference bike:
- Residuals for the reference bike should be centered near zero throughout its period.
- Rolling 30-day mean of residuals colored by bike — look for a step-change at the transition.
- Check extrapolation drift: compare mean predicted values before/after the last reference-bike ride.
- Spline df tuning: run at df=3, 5, 7 and pick the flattest residual-vs-time plot.

**Red flags:**
- Residuals trending up during reference bike period → underfitting, increase df
- Predicted values drifting implausibly after last reference ride → spline extrapolating badly, cap or use linear tail

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.database import init_db, load_efforts, load_segments, load_bikes
from src.bike_delta import (
    prepare_delta_dataset, get_paired_segments,
    fit_baseline_model, compute_residuals,
)
from src.analytics import filter_outliers_by_power_speed

sns.set_theme(style='whitegrid', palette='Set2')
print('Imports OK')

In [ ]:
# ── Load and prepare data ─────────────────────────────────────────────────────
init_db()
raw_efforts = load_efforts()
segments = load_segments()
bikes_df = load_bikes()
bikes_dict = dict(zip(bikes_df['gear_id'], bikes_df['name'])) if not bikes_df.empty else {}

power_efforts = raw_efforts[raw_efforts['average_watts'].notna()].copy()
df = prepare_delta_dataset(power_efforts, segments, bikes_dict)
df, _ = filter_outliers_by_power_speed(df, z_threshold=2.0)

all_bikes = df['bike_name'].dropna().unique().tolist()
print(f'Bikes: {all_bikes}')
print(f'Date range: {df["ride_index"].min():.0f} – {df["ride_index"].max():.0f} days')

In [ ]:
# ── Configure ─────────────────────────────────────────────────────────────────
# Set these to match your data
REF_BIKE = all_bikes[0]          # Reference bike — typically your oldest / most-ridden one
SPLINE_DF = 5                    # Starting df; we'll compare 3, 5, 7 below
MIN_EFFORTS = 3

print(f'Reference bike: {REF_BIKE}')

paired = get_paired_segments(df, all_bikes, min_efforts=MIN_EFFORTS)
print(f'Paired segments: {len(paired)}')

## 2.1 — Fit baseline at default df and inspect residuals

In [ ]:
model, design_info = fit_baseline_model(df, REF_BIKE, spline_df=SPLINE_DF)
df_resid = compute_residuals(df, model, design_info)

print(f'Baseline R² (on reference bike): ', end='')
ref_data = df_resid[df_resid['bike_name'] == REF_BIKE].dropna(subset=['residual'])
ss_res = (ref_data['residual'] ** 2).sum()
ss_tot = ((ref_data['speed_per_cbrt_watt'] - ref_data['speed_per_cbrt_watt'].mean()) ** 2).sum()
print(f'{1 - ss_res / ss_tot:.3f}')

In [ ]:
# ── Residuals vs timestamp, colored by bike ───────────────────────────────────
plot_df = df_resid.dropna(subset=['residual']).copy()
plot_df['_dt'] = pd.to_datetime(plot_df['start_date'], errors='coerce', utc=True).dt.tz_convert(None)

palette = sns.color_palette('Set2', n_colors=len(all_bikes))
bike_color = dict(zip(all_bikes, palette))

fig, ax = plt.subplots(figsize=(14, 5))

for bike in all_bikes:
    bdata = plot_df[plot_df['bike_name'] == bike].sort_values('_dt')
    ax.scatter(bdata['_dt'], bdata['residual'],
               label=bike, color=bike_color[bike], alpha=0.55, s=30)
    # 30-day rolling mean
    bdata_idx = bdata.set_index('_dt')['residual']
    roll = bdata_idx.rolling('30D', min_periods=3).mean()
    ax.plot(roll.index, roll.values, color=bike_color[bike], lw=2.5,
            linestyle='--', label=f'{bike} 30d avg')

ax.axhline(0, color='grey', linestyle='--', lw=1.5)
ax.set_xlabel('Date')
ax.set_ylabel('Residual (speed/W¹⁄³)')
ax.set_title(f'Residuals vs time — baseline fit on {REF_BIKE} (df={SPLINE_DF})')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print('\nWhat to look for:')
print('  ✅  Reference bike residuals hover near 0 throughout — no trend')
print('  ✅  Level shift visible between bikes at the transition point')
print('  ⚠️  Residuals trending up during reference period → increase spline df')
print('  ⚠️  Predicted values drifting implausibly after last reference ride')

## 2.2 — Spline df sensitivity: df = 3, 5, 7

Run the baseline at three different spline flexibilities and compare the residual pattern
on the reference bike.  Pick the df that produces the flattest within-bike residuals.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax, sdf in zip(axes, [3, 5, 7]):
    try:
        m, di = fit_baseline_model(df, REF_BIKE, spline_df=sdf)
        dr = compute_residuals(df, m, di)
        ref_r = dr[dr['bike_name'] == REF_BIKE].dropna(subset=['residual']).copy()
        ref_r['_dt'] = pd.to_datetime(ref_r['start_date'], errors='coerce', utc=True).dt.tz_convert(None)
        ref_r = ref_r.sort_values('_dt')
        ax.scatter(ref_r['_dt'], ref_r['residual'], alpha=0.6, s=25, label='Effort')
        roll = ref_r.set_index('_dt')['residual'].rolling('30D', min_periods=3).mean()
        ax.plot(roll.index, roll.values, lw=2.5, color='tomato', label='30d avg')
        ax.axhline(0, color='grey', linestyle='--', lw=1)
        ax.set_title(f'df = {sdf}  (RMSE = {ref_r["residual"].std():.4f})')
        ax.set_xlabel('Date')
        ax.legend(fontsize=8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        fig.autofmt_xdate()
    except Exception as e:
        ax.set_title(f'df = {sdf}  — FAILED: {e}')

axes[0].set_ylabel(f'{REF_BIKE} residuals (speed/W¹⁄³)')
fig.suptitle(f'Spline df sensitivity — reference bike: {REF_BIKE}', y=1.02)
plt.tight_layout()
plt.show()

## 2.3 — Extrapolation drift check

After the last reference-bike ride, does the model extrapolate plausibly?  Compare
the mean *predicted* value just before vs just after the last reference-bike ride.

In [ ]:
ref_efforts = df_resid[df_resid['bike_name'] == REF_BIKE].dropna(subset=['start_date']).copy()
ref_efforts['_dt'] = pd.to_datetime(ref_efforts['start_date'], errors='coerce', utc=True).dt.tz_convert(None)

last_ref_date = ref_efforts['_dt'].max()
print(f'Last {REF_BIKE} ride: {last_ref_date:%Y-%m-%d}')

before = df_resid[(df_resid['_dt'] <= last_ref_date) if '_dt' in df_resid.columns
                   else df_resid.index.notna()].copy() if '_dt' not in df_resid.columns else \
         df_resid.copy()

# Attach timestamps to full resid df
df_resid['_dt'] = pd.to_datetime(df_resid['start_date'], errors='coerce', utc=True).dt.tz_convert(None)

window = pd.Timedelta(days=60)
before_mask = (df_resid['_dt'] >= last_ref_date - window) & (df_resid['_dt'] <= last_ref_date)
after_mask  = (df_resid['_dt'] > last_ref_date) & (df_resid['_dt'] <= last_ref_date + window)

pred_before = df_resid.loc[before_mask, 'predicted'].mean()
pred_after  = df_resid.loc[after_mask,  'predicted'].mean()

print(f'Mean predicted (60d before last ref ride): {pred_before:.4f}')
print(f'Mean predicted (60d after  last ref ride): {pred_after:.4f}')
drift = abs(pred_after - pred_before)
print(f'Drift: {drift:.4f}  ({"⚠️  Suspicious" if drift > 0.05 else "✅  OK"})')
print('(Drift > 0.05 suggests spline extrapolation — consider capping or linear tail)')